# Future Crop Challenge — submission: literal CERES simulator

Both crops are driven by a **literal, literature-fixed** CERES process model: GDD phenology -> trapezoid LAI -> Priestley-Taylor PET -> single soil-water bucket (supply-type water channel) -> RUE x intercepted light x water-stress x grain-fill/flowering heat x saturating-N x saturating-CO2 -> harvest index. **No weather parameter is fitted** off the competition data; the only per-location fit is the 2-parameter affine `y_obs = a + b*y_sim` (closed-form ridge, ridge=0.1, intercept exempt), calibrated on all 39 train years (381-419) and applied to the full test window (420-497, CO2 up to ~1108 ppm).

| crop | simulated core (fixed from literature) | per-cell fit |
|---|---|---|
| wheat | CERES-Wheat (C3: saturating CO2 gamma=0.40, grain-fill heat-days >=31 C) | `a + b*s` |
| maize | CERES-Maize (C4: saturating CO2 gamma=0.03, flowering heat-days >=24 C) | `a + b*s` |

**Why.** The CERES family is the sandbox's top group for maize on every held-out diagnostic (`results/sim_match_maize.md`; direct corr(obs,sim) +0.16, 75% of cells) and ties CERES/STICS at the top for wheat. The strict adoption rule (>=0.10 margin on both legs) did **not** name a winner, so this notebook is a leadership test: the leaderboard is the arbiter. Because the simulator is generative (runs over 240-day climate *and* CO2), the test-year CO2 rise to ~1100 ppm flows through the model's own mechanism rather than a hand-fitted term.

**Honesty note.** The sandbox found no simulator version that beats the benchmark cores (`13` wheat / `06` maize) on held-out R2; keep `submission_co2_24_06.ipynb` as the primary CO2 arm.

**Fit protocol.** Model is linear in params once the simulated series is fixed, so per-cell fit is the same closed-form ridge as the rest of the portfolio: `theta = (X^T X + T*ridge*R)^{-1} X^T y`, `R = diag(0,1)`. 

## Data loading

Each feature is one parquet per (variable, crop, split). Row layout is uniform across files and aligned by the row **index = submission ID**. `soil_co2_*.parquet` carries per-row metadata (`year`, `lon`, `lat`, `co2`, `nitrogen`); `pr` is stored in m/day and multiplied by 1000 -> mm/day to match the sandbox cache pipeline. On Kaggle use the default `/kaggle/input/the-future-crop-challenge/`; locally uncomment `data/`.

## The simulator

The inline engine below is a row-vectorised copy of `sandbox/sim/core.py` (no-memory path: every row is one (cell, year), simulated independently over its 240-day window). Embedded here so Kaggle needs only the competition dataset. The single `FAM` dict fixes **every** parameter from the literature (`sandbox/sim/families.py`).

## Fit and predict

For each crop: (1) run the literal simulator over train rows, (2) fit one `(a, b)` ridge vector per grid cell on train years 381-419, (3) run the simulator over test rows (420-497) and predict `a + b*s`, floored at 0. Test rows in cells absent from train fall back to the crop train mean.

## Assemble submission

Merge predictions (keyed by test ID = parquet index) onto the sample submission so every row gets a yield. Defensive gap-fill uses the overall train mean.

## Notes / caveats

- **Extrapolation**: test CO2 ~418 -> ~1108 ppm, far beyond train (341-415). The simulator scales yield by its own saturating CO2 multiplier (C3 wheat gamma=0.40 -> ~+25-28% by the ceiling; C4 maize gamma=0.03 -> near-null direct CO2 effect), the same mechanism the `24` wheat core uses on the weather block.
- **What to expect in the run log**: wheat predictions should sit above the per-cell train level (CO2-driven upward trend); maize slightly below or flat (flowering-window heat offsets the mild CO2 gain).
- **Relationship to the sandbox**: `sim_match.py` (500 cells/crop, 381-411/412-419) measured this affine calibration's held-out R2 at around -0.23..-0.28 for the whole CERES/STICS group; the notebook reproduces that map on the full data and the 78-year test window where the CO2 lever operates.
- **Deterministic & fast**: closed-form per-cell ridge, no optimizer; the 210-day loop is fully vectorised over rows.
- Reproduce locally by setting `DATA_DIR = "data/"`.

In [ ]:
import numpy as np
import pandas as pd
import time

t0 = time.time()

# ---- point DATA_DIR at the local repo for testing ----
DATA_DIR = "/kaggle/input/the-future-crop-challenge/"     # Kaggle
# DATA_DIR = "data/"                                      # local run

RIDGE = 0.1                # ridge weight on params[1:]^2 (intercept exempt)
MIN_TRAIN_YEARS = 3        # per-cell min train rows to fit; fewer -> cell mean
SOW_DAY = 30               # 240-day window = 30d pre-sow + 210d season


In [ ]:
def read_meta(crop, split):
    """Metadata carrier parquet: year, lon, lat, CO2, nitrogen."""
    df = pd.read_parquet(f"{DATA_DIR}/soil_co2_{crop}_{split}.parquet")
    return df[["year", "lon", "lat", "co2", "nitrogen"]]


def read_days(feature, crop, split):
    """Day columns '0'..'239' of one climate feature as float32 (N, 240)."""
    df = pd.read_parquet(f"{DATA_DIR}/{feature}_{crop}_{split}.parquet",
                         columns=[str(i) for i in range(240)])
    return df.to_numpy(dtype=np.float32)


def location_codes(meta_tr, meta_te):
    """Consistent per-cell codes across train and test rows."""
    both = pd.concat([meta_tr[["lon", "lat"]], meta_te[["lon", "lat"]]],
                     ignore_index=True)
    codes = both.groupby(["lon", "lat"], sort=False).ngroup().to_numpy()
    return codes[: len(meta_tr)], codes[len(meta_tr):]


In [ ]:
CERES = dict(
    # ---- family-level (structure) ----
    water_channel="supply", pet="pt", soil_memory="no",
    w_init=0.9, stress_k=4.0, kc=1.0, lai_ref=4.0, whc=150.0, kext=0.55,
    n_half=70.0, co2_c0=350.0, co2_k=300.0,
    # ---- crop blocks (literature-fixed) ----
    heat={"wheat": [31.0, 0.08], "maize": [24.0, 0.10]},
    wheat=dict(gdd_base=0.0, tt_ant=2100.0, tt_mat=2800.0,
               tt_heat_lo=2100.0, tt_heat_hi=9999.0,
               lai_max=5.5, rue=2.9, hi=0.45, co2_gamma=0.40),
    maize=dict(gdd_base=8.0, tt_ant=1050.0, tt_mat=1550.0,
               tt_heat_lo=820.0, tt_heat_hi=1300.0,
               lai_max=5.5, rue=3.6, hi=0.50, co2_gamma=0.03),
)

PET_PT_K = 0.018     # Priestley-Taylor: PET mm/d ~ 0.018 * rsds[W/m2]


def _simulate_rows(tasmax, tasmin, pr, rsds, co2, nitrogen, fam, crop):
    """Literal daily crop-engine, one row = one (cell, year).

    Inputs (N, 240) for tasmax/tasmin/pr[mm]/rsds[W/m2], (N,) co2/nitrogen.
    Returns the simulated grain-yield series (N,). Row-vectorised copy of
    sandbox/sim/core.py (no-memory path): GDD phenology -> trapezoid LAI ->
    Priestley-Taylor PET -> single soil-water bucket (supply channel) ->
    RUE x intercepted light x water-stress x heat-damage x saturating-N x
    saturating-CO2 -> harvest index.
    """
    c = fam[crop]
    n = tasmax.shape[0]

    tmean = 0.5 * (tasmax + tasmin)
    gdd = np.maximum(tmean - c["gdd_base"], 0.0)
    cum = np.cumsum(gdd, axis=1)
    cum_sow = np.maximum(cum - cum[:, SOW_DAY - 1:SOW_DAY], 0.0)

    tt_ant, tt_mat = c["tt_ant"], c["tt_mat"]
    dev = (cum_sow / tt_mat).clip(0.0, 1.0)
    dev_ant = tt_ant / max(tt_mat, 1e-6)
    lai = np.minimum(dev / max(dev_ant, 1e-6),
                     np.clip((1.0 - dev) / max(1.0 - dev_ant, 1e-6), 0.0, 1.0))
    lai = np.minimum(lai, 1.0) * c["lai_max"]

    d = co2 - fam["co2_c0"]
    h = np.clip(d / (d + fam["co2_k"]), 0.0, None)
    co2_mult = (1.0 + c["co2_gamma"] * h).clip(None, 1.0 + c["co2_gamma"] * 0.9)
    n_mult = 1.0 - np.exp(-nitrogen / fam["n_half"])

    whc = fam["whc"]
    w = np.full(n, whc * fam["w_init"])
    ass = np.zeros(n)
    heat_thr, heat_k = fam["heat"][crop]
    for d in range(SOW_DAY, 240):
        kc = PET_PT_K * rsds[:, d] * fam["kc"] * np.clip(lai[:, d] / fam["lai_ref"], 0.05, 1.0)
        fa = np.clip(w / whc, 0.0, 1.0)
        wstress = (1.0 - np.exp(-fam["stress_k"] * fa)).clip(0.0, 1.0)
        et = kc * wstress
        in_win = (cum_sow[:, d] >= c["tt_heat_lo"]) & (cum_sow[:, d] <= c["tt_heat_hi"])
        dmg = np.where(in_win, np.maximum(tmean[:, d] - heat_thr, 0.0), 0.0)
        hstress = 1.0 / (1.0 + heat_k * dmg)
        fpar = 1.0 - np.exp(-fam["kext"] * lai[:, d])
        par = 0.5 * rsds[:, d] * (86400.0 / 4.18e6)
        ass += c["rue"] * fpar * par * wstress * hstress * co2_mult * n_mult
        w = np.clip(w + pr[:, d] - et, 0.0, whc)
    return c["hi"] * ass


In [ ]:
FAM = CERES

def make_design(crop, tasmax, tasmin, pr_mm, rsds, co2, nitrogen):
    """Single feature: the literature-fixed simulated yield series."""
    s = _simulate_rows(tasmax, tasmin, pr_mm, rsds, co2, nitrogen, FAM, crop)
    return np.column_stack([np.ones(len(s)), s])


def fit_cells(D_tr, y_tr, loc, ridge=RIDGE):
    """Per-cell closed-form ridge y ~= a + b*s (intercept exempt, p=2)."""
    n_locs = loc.max() + 1
    p = D_tr.shape[1]
    R = np.diag([0.0] + [1.0] * (p - 1))
    params = np.zeros((n_locs, p))
    for cc in range(n_locs):
        m = np.where(loc == cc)[0]
        yc = y_tr[m]
        if len(m) < MIN_TRAIN_YEARS:
            params[cc, 0] = float(yc.mean()) if len(m) else np.nan
            continue
        Xc = D_tr[m]
        try:
            params[cc] = np.linalg.solve(Xc.T @ Xc + len(m) * ridge * R,
                                         Xc.T @ yc)
        except np.linalg.LinAlgError:
            params[cc, 0] = float(yc.mean())
    return params


def predict_split(crop, tasmax, tasmin, pr_mm, rsds, co2, nitrogen, params, loc):
    D = make_design(crop, tasmax, tasmin, pr_mm, rsds, co2, nitrogen)
    pred = (D * params[loc]).sum(axis=1)
    return np.maximum(pred, 0.0)


In [ ]:
preds, crop_stats = {}, {}
for crop in ["maize", "wheat"]:
    print(f"== {crop} ==", flush=True)

    meta_tr = read_meta(crop, "train")
    meta_te = read_meta(crop, "test")
    loc_tr, loc_te = location_codes(meta_tr, meta_te)

    rsds = read_days("rsds", crop, "train")
    tasmax = read_days("tasmax", crop, "train")
    tasmin = read_days("tasmin", crop, "train")
    pr = read_days("pr", crop, "train") * 1000.0              # m/day -> mm
    y_tr = pd.read_parquet(f"{DATA_DIR}/train_solutions_{crop}.parquet")["yield"].to_numpy()
    D_tr = make_design(crop, tasmax, tasmin, pr, rsds,
                       meta_tr["co2"].to_numpy(), meta_tr["nitrogen"].to_numpy())
    params = fit_cells(D_tr, y_tr, loc_tr)
    del tasmax, tasmin, pr, rsds, D_tr

    rsds = read_days("rsds", crop, "test")
    tasmax = read_days("tasmax", crop, "test")
    tasmin = read_days("tasmin", crop, "test")
    pr = read_days("pr", crop, "test") * 1000.0
    pred = predict_split(crop, tasmax, tasmin, pr, rsds,
                         meta_te["co2"].to_numpy(), meta_te["nitrogen"].to_numpy(),
                         params, loc_te)
    del tasmax, tasmin, pr, rsds

    seen = set(np.unique(loc_tr).tolist())
    unseen = np.array([c not in seen for c in loc_te])
    if unseen.any():
        crop_mean = float(np.mean(y_tr))
        pred[unseen] = crop_mean
        print(f"  WARNING: {unseen.sum()} test rows in cells with no train data", flush=True)

    preds[crop] = pd.Series(pred, index=meta_te.index, name="yield")
    crop_stats[crop] = dict(n_pred=len(pred),
                            pred_mean=float(pred.mean()),
                            pred_std=float(pred.std()),
                            train_mean=float(np.mean(y_tr)),
                            n_floored=int((pred == 0.0).sum()))
    print(f"  rows={len(pred)}  pred_mean={pred.mean():.3f} "
          f"train_mean={np.mean(y_tr):.3f}  floored_at_0={int((pred == 0).sum())}",
          flush=True)


In [ ]:
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
pred_all = pd.concat([preds["maize"], preds["wheat"]])
sub["yield"] = sub["ID"].map(pred_all)

missing = int(sub["yield"].isna().sum())
if missing:
    sub["yield"] = sub["yield"].fillna(sub["yield"].mean())
    print(f"WARNING: {missing} rows had no prediction, filled with mean", flush=True)

sub.to_csv("submission.csv", index=False)

print("\n= submission =")
print(f"rows: {len(sub)}  NaNs: {missing}  elapsed: {time.time() - t0:.0f}s")
for crop, s in crop_stats.items():
    print(f"{crop:6s} n={s['n_pred']:7d}  pred_mean={s['pred_mean']:.3f}  "
          f"pred_std={s['pred_std']:.3f}  train_mean={s['train_mean']:.3f}  "
          f"floored_at_0={s['n_floored']}")
print(sub.head())
